# Final run

Prepare the selected retrieval configuration and package the final TREC run.

In [ ]:
%cd ..

import gzip
import shutil
from pathlib import Path

from src.bm25 import Bm25Config
from src.dense_st import GteBaseConfig
from src.paths import TEST_QUERIES_PATH
from src.pipeline import ExperimentConfig, run_experiment
from src.rrf_backend import RRFConfig
from src.span_selector import make_span_selector

## Final configuration

In [ ]:
if not TEST_QUERIES_PATH.exists():
    raise FileNotFoundError(f"Final query file not found: {TEST_QUERIES_PATH}")

final_config = ExperimentConfig(
    experiment_name="jlp",
    query_path=TEST_QUERIES_PATH,
    qrels_path=None,
    run_depth=1000,
    backend=RRFConfig(
        children=(Bm25Config(), GteBaseConfig()),
        rrf_k=1,
    ),
    span_selector=make_span_selector(span_size=1),
    rrf_k=1,
    span_fusion_method="comb_sum",
    span_score_normalization="none",
    span_fusion_rank_cutoff=None,
)

final_config

## Run retrieval

In [ ]:
final_experiment_dir = run_experiment(final_config)
run_path = final_experiment_dir / "run" / "run.trec"

print(f"Experiment directory: {final_experiment_dir}")
print(f"Run path: {run_path}")
print(f"Run exists: {run_path.exists()}")

In [ ]:
submission_path = final_experiment_dir / "run" / "run.txt.gz"

with run_path.open("rb") as source_handle, gzip.open(submission_path, "wb") as target_handle:
    shutil.copyfileobj(source_handle, target_handle)

print(f"Submission path: {submission_path}")
print(f"Submission size bytes: {submission_path.stat().st_size}")